# Renderer luminance gate — MuJoCo 3.3.2 vs 3.3.3 on LIBERO-Object

**This notebook is a GATE, not an experiment.** It answers one question for about a dollar and
no GPU:

> Does the MuJoCo version change how bright LIBERO-Object renders **the specific task Mohan
> trained on**, by enough to matter?

**If the answer is no, the renderer paper is dead and you stop here, one day in instead of six
weeks in.** That is the point of running it first.

## Why this exists

LIBERO issue #88 (`Lifelong-Robot-Learning/LIBERO#88`, open, 0 comments since 2025-06-19) was
filed by `moojink` — Moo Jin Kim, first author of OpenVLA. Verbatim:

> *"MuJoCo's latest version release (3.3.3) may cause LIBERO-Object images to be rendered
> differently, which is a problem since it creates a distribution shift when testing policies
> trained on the original LIBERO data."*

He reports *darker floors* under 3.3.3, reverted by `pip install mujoco==3.3.2`. Nobody has
quantified it in 15 months.

Separately, our own result: dimming the **training** video to x0.30 beat the clean baseline by
+18pp (0.77 vs 0.59). If eval renders darker than the training video, dimming the training data
moves training toward eval — which would make our flagship result partly a renderer artifact.

**This notebook does not test that.** It only measures whether the renderer difference exists on
our task, which is the precondition for everything else.

## The pre-registered gate — decided BEFORE looking at any number

| Measure | Threshold to PROCEED |
|---|---|
| Mean absolute luminance delta, full agentview frame | **>= 2.0 / 255** (0.8%) |
| Mean absolute luminance delta, lower-third (floor) region | **>= 5.0 / 255** (2.0%) |

**Either one passing is enough to proceed.** Both failing means stop. Writing the threshold down
here, before the run, is the whole discipline — do not adjust it after seeing the output.

## Runtime

CPU only. No GPU. ~10-15 min including installs. Runs in two phases with a runtime restart
between them, because you cannot have two MuJoCo versions live in one kernel.


## Phase selector

Set `PHASE` to `"A"` on the first pass, restart the runtime, set it to `"B"`, run again, then run
the comparison cell at the bottom.


In [ ]:
PHASE = "A"          # "A" -> mujoco 3.3.2   |   "B" -> mujoco 3.3.3   |   "COMPARE" -> analysis only

MUJOCO_VERSION = {"A": "3.3.2", "B": "3.3.3"}.get(PHASE)

TASK_SUITE   = "libero_object"
TASK_NAME    = "pick_up_the_alphabet_soup_and_place_it_in_the_basket"
N_STATES     = 20          # fixed initial states rendered per version
CAMERA       = "agentview"
IMG_H, IMG_W = 256, 256

DRIVE_DIR    = "/content/drive/MyDrive/mhh-renderer-gate"

print("PHASE:", PHASE, "| MuJoCo:", MUJOCO_VERSION)

## 1 · Mount Drive

Frames from phase A have to survive the runtime restart, so they go to Drive, not `/content`.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print("writing to:", DRIVE_DIR)

## 2 · Install LIBERO and pin MuJoCo

The pin is the whole experiment, so it is asserted after install rather than trusted.

**The silent failure mode this guards against:** `pip install mujoco==X` also moves transitive
dependencies. If anything other than MuJoCo differs between phases, a luminance delta is not
attributable to the renderer. The full `pip freeze` is saved so the two phases can be diffed.


In [ ]:
import subprocess, sys

if PHASE in ("A", "B"):
    # LIBERO itself (brings robosuite). Quiet, but errors still surface.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/Lifelong-Robot-Learning/LIBERO.git"], check=True)
    # The pin goes LAST so nothing upgrades it afterwards.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    f"mujoco=={MUJOCO_VERSION}"], check=True)

    import mujoco
    assert mujoco.__version__ == MUJOCO_VERSION, (
        f"PIN FAILED: asked for {MUJOCO_VERSION}, got {mujoco.__version__}. "
        "Stop - the experiment is invalid if this does not hold.")
    print("mujoco pinned OK:", mujoco.__version__)

    freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"],
                            capture_output=True, text=True).stdout
    with open(f"{DRIVE_DIR}/freeze_{PHASE}.txt", "w") as f:
        f.write(freeze)
    print("pip freeze saved:", f"freeze_{PHASE}.txt", "|", len(freeze.splitlines()), "packages")

## 3 · Render the SAME fixed initial states under this MuJoCo version

`task_suite.get_task_init_states(task_id)` returns LIBERO's canonical fixed initial-state array.
Using it (rather than `env.seed()` + `reset()`) is what makes the two phases byte-comparable:
the simulator state is *set*, not sampled, so any pixel difference is the renderer and nothing
else.

> Note for anyone reusing this: LIBERO's registered eval environment does **not** call
> `set_init_state()` - only its `__main__` demo block does. That is a separate finding and the
> reason a paired comparison on the stock eval path is not trustworthy. Here we set the states
> deliberately.


In [ ]:
import numpy as np

if PHASE in ("A", "B"):
    from libero.libero import benchmark, get_libero_path
    from libero.libero.envs import OffScreenRenderEnv

    bench   = benchmark.get_benchmark_dict()[TASK_SUITE]()
    names   = [bench.get_task(i).name for i in range(bench.n_tasks)]
    task_id = names.index(TASK_NAME)
    task    = bench.get_task(task_id)
    print("task", task_id, ":", task.name)

    bddl = os.path.join(get_libero_path("bddl_files"),
                        task.problem_folder, task.bddl_file)
    env = OffScreenRenderEnv(bddl_file_name=bddl,
                             camera_heights=IMG_H, camera_widths=IMG_W)
    env.seed(0)

    init_states = bench.get_task_init_states(task_id)
    n = min(N_STATES, len(init_states))
    print("fixed initial states available:", len(init_states), "| rendering:", n)

    frames = []
    for i in range(n):
        env.reset()
        obs = env.set_init_state(init_states[i])
        frames.append(np.asarray(obs[f"{CAMERA}_image"], dtype=np.uint8))
    env.close()

    frames = np.stack(frames)                      # (n, H, W, 3)
    np.save(f"{DRIVE_DIR}/frames_{PHASE}.npy", frames)
    print("saved frames:", frames.shape, "dtype", frames.dtype)

## 4 · STOP — restart the runtime

Phase A is done. Now:

1. **Runtime -> Restart session**
2. Set `PHASE = "B"` in cell 1
3. Run cells 1-3 again
4. Then set `PHASE = "COMPARE"` and run the cell below

A restart is required because a live MuJoCo cannot be swapped under an imported module.


## 5 · Compare — the gate

Luminance is ITU-R BT.601: `0.299R + 0.587G + 0.114B`, on the raw 0-255 frames.

The floor region is taken as the lower third of the frame, which is where issue #88 reports the
change. That is an approximation of "floor", stated so nobody reads it as a segmentation.


In [ ]:
import numpy as np, os, csv

def luma(x):
    x = x.astype(np.float64)
    return 0.299 * x[..., 0] + 0.587 * x[..., 1] + 0.114 * x[..., 2]

A = np.load(f"{DRIVE_DIR}/frames_A.npy")
B = np.load(f"{DRIVE_DIR}/frames_B.npy")
assert A.shape == B.shape, f"shape mismatch {A.shape} vs {B.shape} - the states did not match"

LA, LB = luma(A), luma(B)
floor  = slice(int(LA.shape[1] * 2 / 3), None)      # lower third

full_delta  = float(np.abs(LB - LA).mean())
floor_delta = float(np.abs(LB[:, floor] - LA[:, floor]).mean())
signed_full = float((LB - LA).mean())               # negative => 3.3.3 is DARKER
per_state   = np.abs(LB - LA).reshape(len(LA), -1).mean(axis=1)

GATE_FULL, GATE_FLOOR = 2.0, 5.0
passed = (full_delta >= GATE_FULL) or (floor_delta >= GATE_FLOOR)

print(f"states compared        : {len(LA)}")
print(f"mean |delta| full frame: {full_delta:7.3f} / 255   (gate >= {GATE_FULL})")
print(f"mean |delta| floor     : {floor_delta:7.3f} / 255   (gate >= {GATE_FLOOR})")
print(f"signed mean delta      : {signed_full:+7.3f}        (negative = 3.3.3 darker)")
print(f"per-state |delta| range: {per_state.min():.3f} .. {per_state.max():.3f}")
print()
print("GATE:", "PASS - build the 2x2" if passed else "FAIL - stop, the renderer paper is dead")

with open(f"{DRIVE_DIR}/luminance_gate.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["metric", "value", "units"])
    w.writerow(["n_states", len(LA), "count"])
    w.writerow(["mean_abs_delta_full", round(full_delta, 4), "luma_levels_0_255"])
    w.writerow(["mean_abs_delta_floor", round(floor_delta, 4), "luma_levels_0_255"])
    w.writerow(["signed_mean_delta_full", round(signed_full, 4), "luma_levels_0_255"])
    w.writerow(["gate_threshold_full", GATE_FULL, "luma_levels_0_255"])
    w.writerow(["gate_threshold_floor", GATE_FLOOR, "luma_levels_0_255"])
    w.writerow(["gate_passed", passed, "bool"])
    for i, d in enumerate(per_state):
        w.writerow([f"state_{i:02d}_abs_delta", round(float(d), 4), "luma_levels_0_255"])
print("wrote", f"{DRIVE_DIR}/luminance_gate.csv")

## 6 · Dependency diff — run this before believing the number above

If anything other than `mujoco` differs between the two phases, the luminance delta is not
cleanly attributable to the renderer. This is the check the idea loop explicitly asked for.


In [ ]:
a = dict(l.split("==", 1) for l in open(f"{DRIVE_DIR}/freeze_A.txt").read().splitlines() if "==" in l)
b = dict(l.split("==", 1) for l in open(f"{DRIVE_DIR}/freeze_B.txt").read().splitlines() if "==" in l)

diff = {k: (a.get(k), b.get(k)) for k in set(a) | set(b) if a.get(k) != b.get(k)}
for k, (va, vb) in sorted(diff.items()):
    flag = "  <- the intended change" if k == "mujoco" else "  <-- CONFOUND"
    print(f"{k:35s} A={va}  B={vb}{flag}")

if set(diff) == {"mujoco"}:
    print("\nCLEAN: mujoco is the only difference. The delta is attributable to the renderer.")
else:
    print(f"\nNOT CLEAN: {len(diff)} packages differ. Pin the extras and re-run before using this number.")

## 7 · The three plots the paper needs

Only worth running if the gate passed.


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(15, 4))

ax[0].imshow(A[0]); ax[0].set_title("MuJoCo 3.3.2"); ax[0].axis("off")
ax[1].imshow(B[0]); ax[1].set_title("MuJoCo 3.3.3"); ax[1].axis("off")
d = (LB[0] - LA[0])
m = np.abs(d).max() or 1.0
im = ax[2].imshow(d, cmap="coolwarm", vmin=-m, vmax=m)
ax[2].set_title("luminance difference (3.3.3 - 3.3.2)"); ax[2].axis("off")
fig.colorbar(im, ax=ax[2], fraction=0.046)
plt.tight_layout(); plt.savefig(f"{DRIVE_DIR}/plot1_side_by_side.png", dpi=150); plt.show()

plt.figure(figsize=(7, 4))
plt.hist(LA.ravel(), bins=64, alpha=0.6, label="3.3.2", density=True)
plt.hist(LB.ravel(), bins=64, alpha=0.6, label="3.3.3", density=True)
plt.xlabel("luminance (0-255)"); plt.ylabel("density"); plt.legend()
plt.title("Pixel luminance distribution, all fixed initial states")
plt.tight_layout(); plt.savefig(f"{DRIVE_DIR}/plot2_luma_hist.png", dpi=150); plt.show()

plt.figure(figsize=(7, 4))
plt.bar(range(len(per_state)), per_state)
plt.axhline(GATE_FULL, ls="--", label=f"gate = {GATE_FULL}")
plt.xlabel("fixed initial state"); plt.ylabel("mean |luminance delta|"); plt.legend()
plt.title("Per-state renderer delta")
plt.tight_layout(); plt.savefig(f"{DRIVE_DIR}/plot3_per_state.png", dpi=150); plt.show()